# A2：在想象中行动

我们先用小数据训练 world model，再从真实 posterior state 出发想象 5 步。Actor 选择动作，Critic 学习 TD-λ target。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import torch
from hwm.data import make_pixelworld_dataset
from hwm.neural import (
    Actor, Critic, RSSMState, TinyWorldModel, batch_from_episodes,
    imagine, lambda_returns, world_model_loss,
)
torch.manual_seed(1)

## 1. 先准备一台能运行的 world model

这里仍是 smoke，不把 10 次更新写成训练完成。

In [ ]:
episodes = make_pixelworld_dataset(num_episodes=4, length=8, seed=1)
batch = batch_from_episodes(episodes, sequence_length=8)
observations, actions, rewards, dones = batch
model = TinyWorldModel()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
for _ in range(10):
    optimizer.zero_grad()
    wm_loss, _, outputs = world_model_loss(model, *batch)
    wm_loss.backward()
    optimizer.step()
print('world model smoke loss:', round(float(wm_loss.detach()), 4))

## 2. 从真实 posterior 取一个想象起点

In [ ]:
with torch.no_grad():
    outputs = model(observations, actions, sample=False)
posterior = outputs['posterior']
start = RSSMState(
    posterior.deterministic[:, -1].detach(),
    posterior.stochastic[:, -1].detach(),
    posterior.mean[:, -1].detach(),
    posterior.std[:, -1].detach(),
)
print('起点 feature:', tuple(start.feature.shape))
assert start.feature.shape == (4, 80)

## 3. Actor 提动作，RSSM 推演未来

这一段不再读取真实 observation。

In [ ]:
actor = Actor()
critic = Critic()
imagined = imagine(model, actor, start, horizon=5)
print('imagined features:', tuple(imagined['features'].shape))
print('actions:', imagined['actions'][0].tolist())
print('predicted rewards:', [
    round(x, 3) for x in imagined['rewards'][0].detach().tolist()
])
assert imagined['actions'].shape == (4, 5)

## 4. Critic 与 TD-λ

Critic 估计 imagined state 的 value。TD-λ 把短期 reward 与后续 value 混合成训练目标。

In [ ]:
values = critic(imagined['features'].detach())
returns = lambda_returns(
    imagined['rewards'].detach(),
    imagined['continues'].detach(),
    values.detach(),
    values[:, -1].detach(),
)
print('values shape:', tuple(values.shape))
print('TD-lambda target:', [round(x, 3) for x in returns[0].tolist()])
assert returns.shape == values.shape

## 5. 各更新一次 Actor 与 Critic

教学版 Actor 使用 REINFORCE 形式：高 return 动作提高 log probability。完整 Dreamer 还会研究 dynamics gradient 与离散直通。

In [ ]:
actor_optimizer = torch.optim.Adam(actor.parameters(), lr=1e-3)
critic_optimizer = torch.optim.Adam(critic.parameters(), lr=1e-3)

actor_before = next(actor.parameters()).detach().clone()
actor_loss = -(imagined['log_probs'] * returns.detach()).mean()
actor_optimizer.zero_grad()
actor_loss.backward()
actor_optimizer.step()

critic_loss = torch.nn.functional.mse_loss(values, returns.detach())
critic_optimizer.zero_grad()
critic_loss.backward()
critic_optimizer.step()

print('actor loss:', round(float(actor_loss.detach()), 4))
print('critic loss:', round(float(critic_loss.detach()), 4))
print('Actor 参数改变:', bool(torch.any(
    actor_before != next(actor.parameters()).detach()
)))
assert torch.any(actor_before != next(actor.parameters()).detach())

## 小结

- [ ] Imagination 从真实 posterior state 开始。
- [ ] Actor 提动作，RSSM prior 预测下一 latent。
- [ ] Reward 与 continue heads 给 imagined trajectory 提供训练信号。
- [ ] Critic 用 TD-λ 学 value，Actor 提高高回报动作概率。
- [ ] 一次参数更新只证明训练接口连通，真实能力必须由 PA1-A 环境 return 检查。